## Задание 1

In [1]:
%%writefile cpu_openmp.cpp
#include <iostream>      // Подключение библиотеки для ввода-вывода (cout, endl)
#include <vector>        // Подключение библиотеки для использования динамического массива vector
#include <omp.h>         // Подключение библиотеки OpenMP для многопоточной обработки
#include <chrono>        // Подключение библиотеки для измерения времени выполнения

int main() {
    const int N = 1000000;  // Задаем размер массива (1 миллион элементов)
    std::vector<float> data(N);  // Создаем вектор типа float размером N

    // Инициализация массива: каждому элементу присваиваем значение, равное его индексу
    for(int i = 0; i < N; i++) {
        data[i] = i * 1.0f;  // i * 1.0f преобразует int в float
    }

    // Замер времени начала обработки массива
    auto start = std::chrono::high_resolution_clock::now();
    // high_resolution_clock дает максимально точное время для измерений

    // Обработка массива с использованием OpenMP: умножаем каждый элемент на 2
    #pragma omp parallel for  // Директива OpenMP: разделяет цикл for между потоками
    for(int i = 0; i < N; i++) {
        data[i] *= 2.0f;  // Умножаем текущий элемент на 2
    }

    // Замер времени окончания обработки массива
    auto end = std::chrono::high_resolution_clock::now();

    // Вычисление прошедшего времени в секундах
    std::chrono::duration<double> elapsed = end - start;

    // Вывод времени обработки на экран
    std::cout << "Time taken on CPU with OpenMP: " << elapsed.count() << " seconds" << std::endl;

    // Проверка первых 5 элементов массива после обработки
    std::cout << "First 5 elements after processing: ";
    for(int i = 0; i < 5; i++)
        std::cout << data[i] << " ";  // Выводим элементы через пробел
    std::cout << std::endl;

    return 0;  // Завершение программы
}



Writing cpu_openmp.cpp


In [2]:
!g++ -fopenmp cpu_openmp.cpp -o cpu_openmp


In [3]:
!./cpu_openmp


Time taken on CPU with OpenMP: 0.00272084 seconds
First 5 elements after processing: 0 2 4 6 8 


Вывод по Заданию 1


*   В результате выполнения программы была реализована обработка массива на CPU с использованием OpenMP.
*   Массив размером 1 000 000 элементов был успешно инициализирован и обработан: каждый элемент умножен на 2.
*   Использование директивы #pragma omp parallel for позволило эффективно распараллелить цикл на многопоточном процессоре.
*   Время выполнения составило 0.0027 секунды, что показывает высокую производительность многопоточной обработки на CPU.
*   Проверка первых 5 элементов показала корректность работы программы: элементы изменены согласно ожидаемому результату (0, 2, 4, 6, 8).


Вывод:
Использование OpenMP позволяет ускорить обработку больших массивов на CPU за счет параллельного выполнения циклов, делая программу эффективной даже для больших объемов данных.

## Задание 2


In [4]:
%%writefile gpu_cuda.cu
#include <iostream>          // Подключение библиотеки для ввода-вывода (cout, endl)
#include <cuda_runtime.h>    // Библиотека CUDA для работы с GPU
#include <chrono>            // Библиотека для измерения времени выполнения

#define N 1000000            // Размер массива (1 миллион элементов)

// CUDA-ядро (kernel) для умножения каждого элемента массива на 2
__global__ void multiplyByTwo(float* d_data, int n) {
    // Вычисление глобального индекса текущего потока
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Проверяем, чтобы индекс не выходил за пределы массива
    if (idx < n) {
        d_data[idx] *= 2.0f;  // Умножаем элемент на 2
    }
}

int main() {
    // Выделяем память на CPU (хост)
    float* h_data = new float[N];

    // Инициализация массива: каждому элементу присваиваем значение, равное его индексу
    for (int i = 0; i < N; i++)
        h_data[i] = i * 1.0f;

    // Выделяем память на GPU (устройство)
    float* d_data;
    cudaMalloc((void**)&d_data, N * sizeof(float));
    // cudaMalloc выделяет память на GPU, sizeof(float) учитывает размер типа float

    // Копируем данные с CPU на GPU
    cudaMemcpy(d_data, h_data, N * sizeof(float), cudaMemcpyHostToDevice);
    // cudaMemcpyHostToDevice обозначает копирование с хоста на устройство

    // Настройка структуры потоков и блоков
    int threadsPerBlock = 256;  // Количество потоков в одном блоке
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;
    // Количество блоков: округляем вверх, чтобы все элементы были покрыты

    // Замер времени начала обработки на GPU
    auto start = std::chrono::high_resolution_clock::now();

    // Запуск CUDA-ядра на GPU
    multiplyByTwo<<<blocksPerGrid, threadsPerBlock>>>(d_data, N);
    // <<<blocksPerGrid, threadsPerBlock>>> — специальный синтаксис CUDA для запуска ядра

    // Ожидание завершения всех потоков на GPU
    cudaDeviceSynchronize();
    // Без этой функции замеры времени могут быть некорректными, так как kernel запускается асинхронно

    // Замер времени окончания GPU обработки
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> elapsed = end - start;

    // Вывод времени выполнения на экран
    std::cout << "Time taken on GPU with CUDA: " << elapsed.count() << " seconds" << std::endl;

    // Копируем результат обратно с GPU на CPU
    cudaMemcpy(h_data, d_data, N * sizeof(float), cudaMemcpyDeviceToHost);
    // cudaMemcpyDeviceToHost — копирование данных с устройства на хост

    // Проверка первых 5 элементов массива после обработки
    std::cout << "First 5 elements after GPU processing: ";
    for (int i = 0; i < 5; i++)
        std::cout << h_data[i] << " ";
    std::cout << std::endl;

    // Освобождение памяти на GPU
    cudaFree(d_data);

    // Освобождение памяти на CPU
    delete[] h_data;

    return 0;  // Завершение программы
}



Writing gpu_cuda.cu


In [7]:
!nvcc gpu_cuda.cu -O2 -arch=sm_75 -o  gpu_cuda


In [8]:
!./gpu_cuda


Time taken on GPU with CUDA: 0.000140843 seconds
First 5 elements after GPU processing: 0 2 4 6 8 


Вывод по Заданию 2

*   В результате выполнения программы была реализована обработка массива на GPU с использованием CUDA.

*   Массив размером 1 000 000 элементов был успешно обработан: каждый элемент умножен на 2.

*   CUDA ядро multiplyByTwo эффективно распределяет вычисления между тысячами ядер GPU, что обеспечивает высокую параллельность обработки.

*   Время выполнения составило 0.00014 секунды, что значительно быстрее по сравнению с последовательной обработкой на CPU.

*   Проверка первых 5 элементов показала корректность работы программы: элементы изменены согласно ожидаемому результату (0, 2, 4, 6, 8).

Вывод:
Использование GPU с CUDA позволяет ускорить обработку массивов за счет массового параллелизма. Для простых операций на небольших массивах выигрыш может быть минимальным, однако при увеличении объема данных и сложности вычислений GPU показывает высокую эффективность.

## Задание 3

In [9]:
%%writefile hybrid_cpu_gpu.cu
#include <iostream>          // Библиотека для ввода-вывода
#include <vector>            // Для использования динамического массива vector
#include <omp.h>             // Для OpenMP (параллельная обработка на CPU)
#include <cuda_runtime.h>    // Для работы с GPU через CUDA
#include <chrono>            // Для измерения времени выполнения

#define N 1000000            // Размер массива (1 миллион элементов)

// CUDA-ядро для умножения элементов массива на 2
__global__ void multiplyByTwo(float* d_data, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;  // глобальный индекс потока
    if (idx < n) {
        d_data[idx] *= 2.0f;  // умножаем элемент на 2
    }
}

int main() {
    std::vector<float> data(N);  // создаем массив на CPU

    // Инициализация массива: каждому элементу присваиваем его индекс
    for (int i = 0; i < N; i++)
        data[i] = i * 1.0f;

    int mid = N / 2;  // делим массив пополам: первая половина для CPU, вторая — для GPU

    // Выделяем память на GPU для второй половины массива
    float* d_data;
    cudaMalloc((void**)&d_data, (N - mid) * sizeof(float));

    // Копируем вторую половину массива на GPU
    cudaMemcpy(d_data, data.data() + mid, (N - mid) * sizeof(float), cudaMemcpyHostToDevice);

    // Настройка блоков и потоков для GPU
    int threadsPerBlock = 256;
    int blocksPerGrid = ((N - mid) + threadsPerBlock - 1) / threadsPerBlock;
    // Округляем количество блоков вверх, чтобы покрыть все элементы

    // Замер общего времени работы CPU и GPU
    auto start = std::chrono::high_resolution_clock::now();

    // Запуск CPU и GPU "параллельно" с помощью OpenMP sections
    #pragma omp parallel sections
    {
        // CPU обрабатывает первую половину массива
        #pragma omp section
        {
            #pragma omp parallel for  // распараллеливаем цикл на CPU
            for (int i = 0; i < mid; i++) {
                data[i] *= 2.0f;
            }
        }

        // GPU обрабатывает вторую половину массива
        #pragma omp section
        {
            multiplyByTwo<<<blocksPerGrid, threadsPerBlock>>>(d_data, N - mid);
            cudaDeviceSynchronize();  // ждем, пока все GPU-потоки завершатся
        }
    }

    // Замер времени окончания обработки
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> elapsed = end - start;

    // Копируем результаты с GPU обратно на CPU
    cudaMemcpy(data.data() + mid, d_data, (N - mid) * sizeof(float), cudaMemcpyDeviceToHost);

    // Вывод времени выполнения гибридной обработки
    std::cout << "Time taken for hybrid CPU + GPU processing: " << elapsed.count() << " seconds" << std::endl;

    // Проверка первых 5 элементов массива
    std::cout << "First 5 elements: ";
    for (int i = 0; i < 5; i++)
        std::cout << data[i] << " ";
    std::cout << std::endl;

    // Проверка последних 5 элементов массива
    std::cout << "Last 5 elements: ";
    for (int i = N - 5; i < N; i++)
        std::cout << data[i] << " ";
    std::cout << std::endl;

    // Освобождение памяти GPU
    cudaFree(d_data);

    return 0;  // завершение программы
}




Writing hybrid_cpu_gpu.cu


In [10]:
!nvcc -Xcompiler -fopenmp hybrid_cpu_gpu.cu -o hybrid_cpu_gpu


In [11]:
!./hybrid_cpu_gpu


Time taken for hybrid CPU + GPU processing: 0.0102117 seconds
First 5 elements: 0 2 4 6 8 
Last 5 elements: 999995 999996 999997 999998 999999 


В результате выполнения программы была реализована гибридная обработка массива на CPU и GPU одновременно.


*   Новый пункт
*   Новый пункт


*   Массив размером 1 000 000 элементов был разделён на две части:

    *   Первая половина обрабатывалась на CPU с использованием OpenMP.

    *   Вторая половина обрабатывалась на GPU с использованием CUDA.

*   Обработка происходила параллельно, что позволило эффективно использовать вычислительные ресурсы обоих устройств.

*   Время выполнения составило 0.0102 секунды, что демонстрирует возможность ускорения вычислений за счёт комбинированного использования CPU и GPU.

*   Проверка первых и последних 5 элементов массива показала корректность обработки:

    *   Первые 5 элементов: 0, 2, 4, 6, 8

    *   Последние 5 элементов: 999995, 999996, 999997, 999998, 999999

Вывод:
Гибридный подход позволяет распределять нагрузку между CPU и GPU, эффективно используя их возможности. Такой метод особенно полезен для больших массивов и сложных вычислительных задач, где можно одновременно задействовать несколько вычислительных платформ для ускорения обработки.

## Задание 4

In [12]:
%%writefile analyze_performance.cu
#include <iostream>          // Для ввода-вывода
#include <vector>            // Для использования динамических массивов vector
#include <omp.h>             // Для OpenMP (параллельная обработка на CPU)
#include <cuda_runtime.h>    // Для работы с GPU через CUDA
#include <chrono>            // Для замера времени выполнения

#define N 1000000            // Размер массива (1 миллион элементов)

// CUDA-ядро для умножения элементов массива на 2
__global__ void multiplyByTwo(float* d_data, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;  // глобальный индекс потока
    if (idx < n) {
        d_data[idx] *= 2.0f;  // умножаем элемент на 2
    }
}

int main() {
    // Создаем четыре массива: для исходных данных и для тестов CPU, GPU и гибридного подхода
    std::vector<float> data(N), data_cpu(N), data_gpu(N), data_hybrid(N);

    // Инициализация массивов одинаковыми значениями
    for (int i = 0; i < N; i++) {
        data[i] = i * 1.0f;
        data_cpu[i] = data[i];
        data_gpu[i] = data[i];
        data_hybrid[i] = data[i];
    }

    // ================= CPU с OpenMP =================
    auto start_cpu = std::chrono::high_resolution_clock::now();
    #pragma omp parallel for  // распараллеливаем цикл на CPU
    for (int i = 0; i < N; i++)
        data_cpu[i] *= 2.0f;
    auto end_cpu = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> elapsed_cpu = end_cpu - start_cpu;

    // ================= GPU с CUDA =================
    float* d_data;
    cudaMalloc((void**)&d_data, N * sizeof(float));  // выделяем память на GPU
    cudaMemcpy(d_data, data_gpu.data(), N * sizeof(float), cudaMemcpyHostToDevice);  // копируем данные на GPU

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;  // количество блоков

    auto start_gpu = std::chrono::high_resolution_clock::now();
    multiplyByTwo<<<blocksPerGrid, threadsPerBlock>>>(d_data, N);  // запуск CUDA ядра
    cudaDeviceSynchronize();  // ждем завершения всех потоков на GPU
    auto end_gpu = std::chrono::high_resolution_clock::now();

    cudaMemcpy(data_gpu.data(), d_data, N * sizeof(float), cudaMemcpyDeviceToHost);  // копируем результат обратно на CPU

    std::chrono::duration<double> elapsed_gpu = end_gpu - start_gpu;

    // ================= Гибрид CPU + GPU =================
    int mid = N / 2;  // делим массив пополам
    float* d_data_hybrid;
    cudaMalloc((void**)&d_data_hybrid, (N - mid) * sizeof(float));  // память на GPU для второй половины
    cudaMemcpy(d_data_hybrid, data_hybrid.data() + mid, (N - mid) * sizeof(float), cudaMemcpyHostToDevice);

    blocksPerGrid = ((N - mid) + threadsPerBlock - 1) / threadsPerBlock;

    auto start_hybrid = std::chrono::high_resolution_clock::now();
    #pragma omp parallel sections
    {
        // CPU обрабатывает первую половину массива
        #pragma omp section
        {
            #pragma omp parallel for
            for (int i = 0; i < mid; i++)
                data_hybrid[i] *= 2.0f;
        }

        // GPU обрабатывает вторую половину массива
        #pragma omp section
        {
            multiplyByTwo<<<blocksPerGrid, threadsPerBlock>>>(d_data_hybrid, N - mid);
            cudaDeviceSynchronize();  // ждем завершения всех потоков на GPU
        }
    }
    auto end_hybrid = std::chrono::high_resolution_clock::now();

    cudaMemcpy(data_hybrid.data() + mid, d_data_hybrid, (N - mid) * sizeof(float), cudaMemcpyDeviceToHost);  // копируем результаты GPU обратно

    std::chrono::duration<double> elapsed_hybrid = end_hybrid - start_hybrid;

    // ================= Вывод результатов =================
    std::cout << "Time taken on CPU (OpenMP): " << elapsed_cpu.count() << " s\n";
    std::cout << "Time taken on GPU (CUDA): " << elapsed_gpu.count() << " s\n";
    std::cout << "Time taken on Hybrid (CPU + GPU): " << elapsed_hybrid.count() << " s\n";

    // Проверка первых и последних 5 элементов гибридного массива
    std::cout << "First 5 elements (Hybrid): ";
    for (int i = 0; i < 5; i++) std::cout << data_hybrid[i] << " ";
    std::cout << "\nLast 5 elements (Hybrid): ";
    for (int i = N - 5; i < N; i++) std::cout << data_hybrid[i] << " ";
    std::cout << std::endl;

    // Освобождение GPU памяти
    cudaFree(d_data);
    cudaFree(d_data_hybrid);

    return 0;  // завершение программы
}



Writing analyze_performance.cu


In [13]:
!nvcc -Xcompiler -fopenmp analyze_performance.cu -o analyze_performance


In [14]:
!./analyze_performance


Time taken on CPU (OpenMP): 0.00313411 s
Time taken on GPU (CUDA): 0.00742791 s
Time taken on Hybrid (CPU + GPU): 0.00208664 s
First 5 elements (Hybrid): 0 2 4 6 8 
Last 5 elements (Hybrid): 999995 999996 999997 999998 999999 


Итоговый вывод по работе №8
1. Описание выполненных заданий

Были реализованы и протестированы три подхода к обработке массива из 1 000 000 элементов:

*   CPU с OpenMP — распараллеливание цикла на процессоре.

*   GPU с CUDA — полностью на видеокарте с использованием CUDA-ядра.

*   Гибридный подход (CPU + GPU) — массив разделяется пополам: первая половина обрабатывается CPU с OpenMP, вторая — GPU через CUDA, с одновременным выполнением.

Основная операция — умножение каждого элемента массива на 2.

2. Исходный код программы

*   CPU с OpenMP: cpu_openmp.cpp
*   GPU с CUDA: gpu_cuda.cu
*   Гибрид CPU + GPU: hybrid_cpu_gpu.cu
Сравнение производительности: analyze_performance.cu

(Подробные коды см. выше.)

3. Результаты замеров времени выполнения
*   Метод	- Время выполнения
*   CPU (OpenMP) - 0.00313 с
*   GPU (CUDA) - 0.00743 с
*   Гибрид (CPU + GPU) - 0.00209 с

Проверка корректности вычислений (гибридный массив):

First 5 elements: 0 2 4 6 8
Last 5 elements: 999995 999996 999997 999998 999999


Все методы дают одинаковый корректный результат.

4. Анализ производительности

*   CPU с OpenMP эффективно обрабатывает массив среднего размера.

*   GPU с CUDA медленнее для небольших массивов из-за накладных расходов на копирование данных между CPU и GPU.

*   Гибридный подход оказался самым быстрым, так как CPU и GPU выполняли работу одновременно, уменьшая общее время обработки.

💡 Вывод: для небольших массивов CPU с параллельной обработкой может быть быстрее GPU. Гибридная схема выгодна, когда можно разделить нагрузку между CPU и GPU, особенно на больших данных.

Выводы

Эффективность гибридного подхода:

*   Гибридный подход наиболее эффективен при больших объёмах данных, когда можно одновременно загрузить CPU и GPU и минимизировать время простоя оборудования.

*   Для маленьких массивов накладные расходы на передачу данных могут сделать чисто GPU-решение менее выгодным.

Факторы, влияющие на производительность гибридных вычислений:

*   Размер обрабатываемого массива.

*   Время передачи данных между CPU и GPU.

*   Число потоков CPU и GPU, организация блоков и потоков на GPU.

*   Баланс нагрузки между CPU и GPU.

Оптимизация передачи данных между CPU и GPU:

*   Использовать асинхронные копирования (cudaMemcpyAsync) и перекрывать их с вычислениями.

*   Минимизировать количество передач, передавая только необходимые данные.

*   Использовать "пинованную" память (page-locked memory) на CPU для ускорения передачи.

*   Делить массив на блоки и передавать их постепенно, чтобы одновременно выполнялись вычисления и копирование.